In [ ]:
class LowLightDataset(Dataset):
    def __init__(self, low_light_dir, normal_dir, transform=None):
        self.low_light_images = sorted(os.listdir(low_light_dir))
        self.normal_images    = sorted(os.listdir(normal_dir))
        self.low_light_dir    = low_light_dir
        self.normal_dir       = normal_dir
        self.transform        = transform

    def __len__(self):
        return len(self.low_light_images)

    def __getitem__(self, idx):
        low_light_path = os.path.join(self.low_light_dir, self.low_light_images[idx])
        normal_path = os.path.join(self.normal_dir, self.normal_images[idx])

        low_light = cv2.imread(low_light_path)
        normal = cv2.imread(normal_path)
        #print(low_light.shape)
        low_light = cv2.cvtColor(low_light, cv2.COLOR_BGR2RGB)
        normal = cv2.cvtColor(normal, cv2.COLOR_BGR2RGB)

        low_light = torch.from_numpy(low_light).permute(2, 0, 1).float() / 255.0
        normal = torch.from_numpy(normal).permute(2, 0, 1).float() / 255.0

        return low_light, normal


In [ ]:
import re
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

import re
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset


class LOLv2Dataset(Dataset):
    IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")

    def __init__(self, low_light_dir, normal_dir, transform=None, strict=True, return_paths=False):
        self.low_dir = Path(low_light_dir)
        self.normal_dir = Path(normal_dir)
        self.transform = transform
        self.strict = strict
        self.return_paths = return_paths

        if not self.low_dir.exists():
            raise FileNotFoundError(f"LOW_DIR no existe: {self.low_dir}")
        if not self.normal_dir.exists():
            raise FileNotFoundError(f"NORMAL_DIR no existe: {self.normal_dir}")

        self.gt_by_id = {}
        for p in self.normal_dir.iterdir():
            if p.is_file() and p.suffix.lower() in self.IMG_EXTS:
                m = re.match(r"^normal(\d+)", p.stem, flags=re.IGNORECASE)
                if m:
                    self.gt_by_id[m.group(1)] = p

        pairs = []
        missing = []

        for p in self.low_dir.iterdir():
            if not (p.is_file() and p.suffix.lower() in self.IMG_EXTS):
                continue

            m = re.match(r"^low(\d+)", p.stem, flags=re.IGNORECASE)
            if not m:
                continue  

            pid = m.group(1)
            gt = self.gt_by_id.get(pid, None)

            if gt is None:
                missing.append(p.name)
                if strict:
                    raise RuntimeError(
                        f"GT no found for {p.name} "
                        f"(find {pid}.* en {self.normal_dir})"
                    )
                else:
                    continue

            pairs.append((p, gt))

        if len(pairs) == 0:
            raise RuntimeError(
                "No pairs found low/normal.\n"
                f"LOW_DIR: {self.low_dir}\n"
                f"NORMAL_DIR: {self.normal_dir}\n"
                "Verify names type low00001.png and normal00001.png."
            )

        if (not strict) and missing:
            print(f"[WARN]  omitted {len(missing)} lows without GT ( 10): {missing[:10]}")

        def _id_from_low(path: Path) -> int:
            m = re.match(r"^low(\d+)", path.stem, flags=re.IGNORECASE)
            return int(m.group(1)) if m else 10**18

        self.pairs = sorted(pairs, key=lambda x: _id_from_low(x[0]))

    def __len__(self):
        return len(self.pairs)

    @staticmethod
    def _read_rgb_uint8(path: Path) -> np.ndarray:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"No pude leer imagen: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img  # HWC uint8

    @staticmethod
    def _to_tensor01(img) -> torch.Tensor:
        if isinstance(img, torch.Tensor):
            t = img.float()
            if t.ndim == 3 and t.shape[0] not in (1, 3):
                # si por alguna razón viene HWC en tensor
                t = t.permute(2, 0, 1)
            if t.max() > 1.5:
                t = t / 255.0
            return t

        if isinstance(img, np.ndarray):
            t = torch.from_numpy(img)
            if t.ndim == 3:
                # HWC -> CHW
                t = t.permute(2, 0, 1)
            t = t.float()
            if t.max() > 1.5:
                t = t / 255.0
            return t

        raise TypeError(f"Not supported in _to_tensor01: {type(img)}")

    def __getitem__(self, idx):
        low_path, gt_path = self.pairs[idx]
        #print(low_path)
        #print(gt_path)

        low = self._read_rgb_uint8(low_path)
        gt = self._read_rgb_uint8(gt_path)

        if self.transform is not None:
            low = self.transform(low)
            gt = self.transform(gt)
        else:
            low = self._to_tensor01(low)
            gt = self._to_tensor01(gt)

        if not isinstance(low, torch.Tensor):
            low = self._to_tensor01(low)
        else:
            low = low.float()
            if low.max() > 1.5:
                low = low / 255.0

        if not isinstance(gt, torch.Tensor):
            gt = self._to_tensor01(gt)
        else:
            gt = gt.float()
            if gt.max() > 1.5:
                gt = gt / 255.0

        if self.return_paths:
            return low, gt, str(low_path), str(gt_path)

        return low, gt